# L4 — Sensitive Data Masking

## What You'll Learn

- How to attach a Lambda interceptor to an AgentCore Gateway as a RESPONSE interception point
- How to apply a Bedrock Guardrail inside a Lambda to anonymize PII in tool responses
- How to extend an existing Gateway IAM role to allow invoking a new Lambda
- How to verify that PII fields (email, address) are masked before reaching the orchestrator

## Overview

Every tool response that passes through AnyCompany's Gateway may contain customer PII — email addresses, shipping addresses, phone numbers. This notebook adds a Lambda interceptor that sits between the tool Lambda and the orchestrator. After each tool call, the interceptor applies the Bedrock Guardrail (created in L2) to anonymize PII in the response body before it reaches the Harness.

## Architecture

```
  Orchestrator (Harness)
       │
       ▼
  AgentCore Gateway
       │  tool call (MCP tools/call)
       ▼
  Lambda Tool (order / refund)
       │  raw response (may contain PII)
       ▼
  ┌─────────────────────────────────────────┐
  │  Lambda Interceptor (RESPONSE point)    │
  │  anycompany-pii-interceptor             │
  │       │                                 │
  │       └──► bedrock-runtime.apply_guardrail
  │            (anonymizes email, address,  │
  │             phone, credit card, SSN)    │
  └─────────────────────────────────────────┘
       │  sanitized response
       ▼
  Orchestrator (Harness)
```

## Steps

1. Load configuration from SSM
2. Load the Bedrock Guardrail ID from SSM
3. Create the IAM role for the interceptor Lambda
4. Deploy the interceptor Lambda and update the Gateway IAM role
5. Attach the interceptor to the Gateway
6. Test PII masking end-to-end
> **Compliance Note**: When handling PII, PHI, or payment card data in production, ensure compliance with applicable regulations (GDPR for EU user data, HIPAA for healthcare data, PCI-DSS for payment card data). This notebook demonstrates technical controls; customers are responsible for implementing additional compliance requirements specific to their data classification and jurisdiction. See AWS compliance resources: https://aws.amazon.com/compliance/


## License Details

Refer to LICENSE file in the main folder for license details.

## Prerequisites

- `1_pluggable_inference_layer.ipynb` completed — `guardrail_id` and `guardrail_version` must be in SSM
- `1_pre-requisites.ipynb` completed — Gateway, Cognito, and `gateway_role_arn` must be in SSM
- `2_order_agent.ipynb` completed — `order-tools` Gateway Target must exist
- `boto3` (installed in the first code cell)

## Step 1: Load Configuration from SSM

Install boto3, import libraries, and load Gateway and Cognito IDs from SSM.

In [ ]:
%pip install --quiet boto3==1.43.0
import boto3, json, time, io, zipfile, os
from botocore.exceptions import ClientError

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
SSM_PREFIX = "/anycompany/agentcore"

sts = boto3.client("sts", region_name=REGION)
ACCOUNT_ID = sts.get_caller_identity()["Account"]

ssm = boto3.client("ssm", region_name=REGION)
iam = boto3.client("iam")
lambda_client = boto3.client("lambda", region_name=REGION)
bedrock = boto3.client("bedrock", region_name=REGION)
agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)

def get_ssm(name, default=None):
    try:
        return ssm.get_parameter(Name=name, WithDecryption=True)["Parameter"]["Value"]
    except ssm.exceptions.ParameterNotFound:
        if default is not None:
            return default
        raise

GATEWAY_ID = get_ssm(f"{SSM_PREFIX}/gateway_id")
GATEWAY_ARN = get_ssm(f"{SSM_PREFIX}/gateway_arn")
COGNITO_USER_POOL_ID = get_ssm(f"{SSM_PREFIX}/cognito_user_pool_id")
COGNITO_CLIENT_ID = get_ssm(f"{SSM_PREFIX}/cognito_client_id")
ORDERS_TABLE = get_ssm(f"{SSM_PREFIX}/dynamodb_orders_table")

print(f"Region:       {REGION}")
print(f"Gateway ID:   {GATEWAY_ID}")
print(f"Gateway ARN:  {GATEWAY_ARN}")
print(f"Cognito Pool: {COGNITO_USER_POOL_ID}")
print(f"Orders Table: {ORDERS_TABLE}")

## Step 2: Load Bedrock Guardrail from SSM

Read the guardrail ID and version published by `1_pluggable_inference_layer.ipynb` (Step 6).

In [ ]:
# Read guardrail created by L2 inference notebook (section 7)
# Prerequisite: run l2-inference/pluggable-inference-layer.ipynb section 7 first
GUARDRAIL_ID = get_ssm(f"{SSM_PREFIX}/guardrail_id")
GUARDRAIL_VERSION = get_ssm(f"{SSM_PREFIX}/guardrail_version")

print(f"Guardrail ID:      {GUARDRAIL_ID}")
print(f"Guardrail Version: {GUARDRAIL_VERSION}")

## Step 3: Create IAM Role for Interceptor Lambda

Create the execution role for the interceptor Lambda with permissions for CloudWatch Logs and `bedrock:ApplyGuardrail`.

In [ ]:
INTERCEPTOR_ROLE_NAME = "AnyCompanyPIIInterceptorLambdaRole"

trust = json.dumps({
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "lambda.amazonaws.com"}, "Action": "sts:AssumeRole"}]
})

perms = json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow", "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents"], "Resource": f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:/aws/lambda/anycompany-pii-interceptor:*"},
        {"Effect": "Allow", "Action": ["bedrock:ApplyGuardrail"], "Resource": f"arn:aws:bedrock:{REGION}:{ACCOUNT_ID}:guardrail/{GUARDRAIL_ID}"},
    ]
})

try:
    r = iam.create_role(RoleName=INTERCEPTOR_ROLE_NAME, AssumeRolePolicyDocument=trust)
    INTERCEPTOR_ROLE_ARN = r["Role"]["Arn"]
    print(f"Created: {INTERCEPTOR_ROLE_ARN}")
except iam.exceptions.EntityAlreadyExistsException:
    INTERCEPTOR_ROLE_ARN = iam.get_role(RoleName=INTERCEPTOR_ROLE_NAME)["Role"]["Arn"]
    print(f"Exists: {INTERCEPTOR_ROLE_ARN}")

iam.put_role_policy(RoleName=INTERCEPTOR_ROLE_NAME, PolicyName="PIIInterceptorPerms", PolicyDocument=perms)
print("Permissions attached.")
time.sleep(10)

## Step 4: Deploy Interceptor Lambda

Package and deploy `anycompany-pii-interceptor` (Python 3.12). The handler applies `bedrock-runtime.apply_guardrail` to every `tools/call` response body before returning it to the orchestrator.

In [ ]:
INTERCEPTOR_CODE = """import json, os, boto3, logging

logger = logging.getLogger()
logger.setLevel(logging.INFO)

bedrock_runtime = boto3.client('bedrock-runtime')

GUARDRAIL_ID = os.environ.get('GUARDRAIL_ID')
GUARDRAIL_VERSION = os.environ.get('GUARDRAIL_VERSION', 'DRAFT')


def mask_pii_with_guardrails(text):
    if not GUARDRAIL_ID:
        return text
    try:
        response = bedrock_runtime.apply_guardrail(
            guardrailIdentifier=GUARDRAIL_ID,
            guardrailVersion=GUARDRAIL_VERSION,
            source='OUTPUT',
            outputScope='FULL',
            content=[{'text': {'text': text}}]
        )
        outputs = response.get('outputs', [])
        if outputs:
            return outputs[0].get('text', text)
        return text
    except Exception as e:
        logger.error(f'Guardrail error: {e}')
        return text


def mask_tool_response(response_body):
    masked = json.loads(json.dumps(response_body))
    if 'result' not in masked or 'content' not in masked.get('result', {}):
        return masked
    for i, item in enumerate(masked['result']['content']):
        if item.get('type') != 'text' or not item.get('text'):
            continue
        text_val = item['text']
        try:
            parsed = json.loads(text_val)
            anonymized_str = mask_pii_with_guardrails(json.dumps(parsed, indent=2))
            # Always keep text as a string — MCP protocol requires content[i].text to be a string
            masked['result']['content'][i]['text'] = anonymized_str
        except (json.JSONDecodeError, TypeError):
            masked['result']['content'][i]['text'] = mask_pii_with_guardrails(text_val)
    return masked


def lambda_handler(event, context):
    logger.info(f'Interceptor invoked - method: {event.get("mcp", {}).get("gatewayRequest", {}).get("body", {}).get("method")}, status: {event.get("mcp", {}).get("gatewayResponse", {}).get("statusCode")}')
    try:
        mcp_data = event.get('mcp', {})
        gateway_response = mcp_data.get('gatewayResponse', {})
        gateway_request = mcp_data.get('gatewayRequest', {})
        response_headers = gateway_response.get('headers', {})
        response_body = gateway_response.get('body', {})
        status_code = gateway_response.get('statusCode', 200)
        method = gateway_request.get('body', {}).get('method', '')

        if method == 'tools/call':
            masked_body = mask_tool_response(response_body)
        else:
            masked_body = response_body

        return {
            'interceptorOutputVersion': '1.0',
            'mcp': {
                'transformedGatewayResponse': {
                    'headers': response_headers,
                    'body': masked_body,
                    'statusCode': status_code
                }
            }
        }
    except Exception as e:
        logger.error(f'Lambda error: {e}', exc_info=True)
        gw = event.get('mcp', {}).get('gatewayResponse', {})
        return {
            'interceptorOutputVersion': '1.0',
            'mcp': {
                'transformedGatewayResponse': {
                    'headers': gw.get('headers', {}),
                    'body': gw.get('body', {}),
                    'statusCode': gw.get('statusCode', 500)
                }
            }
        }
"""

buf = io.BytesIO()
with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("lambda_function.py", INTERCEPTOR_CODE)
zip_bytes = buf.getvalue()

INTERCEPTOR_LAMBDA_NAME = "anycompany-pii-interceptor"

try:
    resp = lambda_client.create_function(
        FunctionName=INTERCEPTOR_LAMBDA_NAME,
        Runtime="python3.12",
        Role=INTERCEPTOR_ROLE_ARN,
        Handler="lambda_function.lambda_handler",
        Code={"ZipFile": zip_bytes},
        Timeout=30,
        MemorySize=256,
        Environment={"Variables": {
            "GUARDRAIL_ID": GUARDRAIL_ID,
            "GUARDRAIL_VERSION": GUARDRAIL_VERSION,
        }},
    )
    INTERCEPTOR_LAMBDA_ARN = resp["FunctionArn"]
    print(f"Created: {INTERCEPTOR_LAMBDA_ARN}")
except lambda_client.exceptions.ResourceConflictException:
    INTERCEPTOR_LAMBDA_ARN = lambda_client.get_function(FunctionName=INTERCEPTOR_LAMBDA_NAME)["Configuration"]["FunctionArn"]
    lambda_client.update_function_code(FunctionName=INTERCEPTOR_LAMBDA_NAME, ZipFile=zip_bytes)
    # Wait for code update to complete before updating configuration
    lambda_client.get_waiter('function_updated_v2').wait(FunctionName=INTERCEPTOR_LAMBDA_NAME)
    lambda_client.update_function_configuration(
        FunctionName=INTERCEPTOR_LAMBDA_NAME,
        Environment={"Variables": {
            "GUARDRAIL_ID": GUARDRAIL_ID,
            "GUARDRAIL_VERSION": GUARDRAIL_VERSION,
        }},
    )
    print(f"Updated: {INTERCEPTOR_LAMBDA_ARN}")

# Allow Gateway to invoke the interceptor
try:
    lambda_client.add_permission(
        FunctionName=INTERCEPTOR_LAMBDA_NAME,
        StatementId="AllowGatewayInterceptorInvoke",
        Action="lambda:InvokeFunction",
        Principal="bedrock-agentcore.amazonaws.com",
    )
    print("Gateway invoke permission added.")
except lambda_client.exceptions.ResourceConflictException:
    print("Permission already exists.")

# Wait for Lambda to be active
for _ in range(20):
    fn = lambda_client.get_function_configuration(FunctionName=INTERCEPTOR_LAMBDA_NAME)
    if fn.get("State") == "Active" and fn.get("LastUpdateStatus", "Successful") in ("Successful", None):
        print("Lambda Active and ready.")
        break
    time.sleep(5)

Extend the Gateway IAM role to allow invoking the interceptor Lambda (the role was created in `1_pre-requisites.ipynb` and only covered the order and refund tool Lambdas).

In [ ]:
GATEWAY_ROLE_ARN = get_ssm(f"{SSM_PREFIX}/gateway_role_arn")
GATEWAY_ROLE_NAME = GATEWAY_ROLE_ARN.split("/")[-1]  # extract role name from ARN
GATEWAY_POLICY_NAME = "AnyCompanyGatewayPermissions"  # inline policy name set in 1_pre-requisites.ipynb

existing_policy = iam.get_role_policy(
    RoleName=GATEWAY_ROLE_NAME,
    PolicyName=GATEWAY_POLICY_NAME,
)["PolicyDocument"]

# Append the interceptor ARN to the InvokeLambdaTargets statement (if not already present).
for stmt in existing_policy["Statement"]:
    if stmt.get("Sid") == "InvokeLambdaTargets":
        resources = stmt["Resource"] if isinstance(stmt["Resource"], list) else [stmt["Resource"]]
        if INTERCEPTOR_LAMBDA_ARN not in resources:
            resources.append(INTERCEPTOR_LAMBDA_ARN)
            stmt["Resource"] = resources
            print(f"Adding {INTERCEPTOR_LAMBDA_ARN} to InvokeLambdaTargets")
        else:
            print(f"Already present: {INTERCEPTOR_LAMBDA_ARN}")
        break
else:
    raise RuntimeError("InvokeLambdaTargets statement not found — was 1_pre-requisites.ipynb run?")

iam.put_role_policy(
    RoleName=GATEWAY_ROLE_NAME,
    PolicyName=GATEWAY_POLICY_NAME,
    PolicyDocument=json.dumps(existing_policy),
)
print("Gateway role updated.")
time.sleep(10)  # IAM propagation

## Step 5: Attach Interceptor to the Gateway

Update the existing Gateway to add the interceptor as a RESPONSE interception point, then wait for READY status.

In [ ]:
# Get current gateway config to preserve existing settings
gw = agentcore_control.get_gateway(gatewayIdentifier=GATEWAY_ID)
print(f"Gateway status: {gw.get('status')}")
print(f"Current interceptors: {len(gw.get('interceptorConfigurations', []))}")

GATEWAY_ROLE_ARN = get_ssm(f"{SSM_PREFIX}/gateway_role_arn")
GATEWAY_NAME = get_ssm(f"{SSM_PREFIX}/gateway_name")
COGNITO_USER_POOL_ID = get_ssm(f"{SSM_PREFIX}/cognito_user_pool_id")
COGNITO_CLIENT_ID = get_ssm(f"{SSM_PREFIX}/cognito_client_id")

# Update gateway to add the interceptor — must include all required fields
try:
    agentcore_control.update_gateway(
        gatewayIdentifier=GATEWAY_ID,
        name=GATEWAY_NAME,
        roleArn=GATEWAY_ROLE_ARN,
        authorizerType="CUSTOM_JWT",
        authorizerConfiguration={
            "customJWTAuthorizer": {
                "discoveryUrl": f"https://cognito-idp.{REGION}.amazonaws.com/{COGNITO_USER_POOL_ID}/.well-known/openid-configuration",
                "allowedAudience": [COGNITO_CLIENT_ID],
            }
        },
        interceptorConfigurations=[
            {
                "interceptor": {
                    "lambda": {
                        "arn": INTERCEPTOR_LAMBDA_ARN
                    }
                },
                "interceptionPoints": ["RESPONSE"],
                "inputConfiguration": {
                    "passRequestHeaders": True
                }
            }
        ],
    )
    print("Gateway updated with PII interceptor.")
except Exception as e:
    print(f"Error updating gateway: {e}")
    raise

# Wait for gateway to be READY
print("Waiting for gateway to be ready...")
for _ in range(30):
    g = agentcore_control.get_gateway(gatewayIdentifier=GATEWAY_ID)
    status = g.get("status")
    if status == "READY":
        print("Gateway READY with interceptor attached.")
        break
    print(f"  {status}...")
    time.sleep(5)

## Step 6: Test PII Masking

Authenticate as `gold_customer`, call `check_order_details` for ORD-10001 via the Gateway, and verify that `contactInfo` (email) and `shippingAddress` are anonymized in the response.

In [ ]:
import boto3, json
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest
import requests as req_lib

# Retrieve the workshop test password from SSM (stored as SecureString).
# get_ssm() passes WithDecryption=True so we get the plaintext back.
USER_PASSWORD = get_ssm(f"{SSM_PREFIX}/user_password")

# Get a Cognito token for testing
# ⚠️ WORKSHOP ONLY — disposable Cognito test user; password fetched from SSM.
# In production, use federated identity (SAML/OIDC) instead of shared passwords.
cognito = boto3.client("cognito-idp", region_name=REGION)
auth_resp = cognito.initiate_auth(
    ClientId=COGNITO_CLIENT_ID,
    AuthFlow="USER_PASSWORD_AUTH",
    AuthParameters={"USERNAME": "gold_customer", "PASSWORD": USER_PASSWORD},
)
test_token = auth_resp["AuthenticationResult"]["IdToken"]
print("Got test token for gold_customer")

GATEWAY_URL = get_ssm(f"{SSM_PREFIX}/gateway_url")

# Find the order-tools target
targets = agentcore_control.list_gateway_targets(gatewayIdentifier=GATEWAY_ID)
order_target = None
for t in targets.get("targets", targets.get("items", [])):
    if "order" in t.get("name", "").lower():
        order_target = t
        break

if not order_target:
    print("Order tools target not found. Make sure 2_order_agent.ipynb has been run.")
else:
    print(f"Found order target: {order_target['name']}")

    # Call check_order_details via Gateway MCP
    mcp_request = {
        "jsonrpc": "2.0",
        "method": "tools/call",
        "id": 1,
        "params": {
            "name": f"{order_target['name']}___check_order_details",
            "arguments": {"order_id": "ORD-10001"}
        }
    }

    response = req_lib.post(
        GATEWAY_URL,
        headers={
            "Authorization": f"Bearer {test_token}",
            "Content-Type": "application/json",
        },
        json=mcp_request,
        timeout=30,
    )

    print(f"Response status: {response.status_code}")
    if response.status_code == 200:
        result = response.json()
        print(json.dumps(result, indent=2))
        print()
        print("Expected: contactInfo (email) and shippingAddress should be anonymized ([EMAIL], [ADDRESS])")
    else:
        print(f"Error: {response.text}")

## Step 7: Try Masking End-to-End in the Chatbot UI

Now that the interceptor is wired into the Gateway, verify masking through the chatbot you launched in `l3-orchestration/5_chatbot_ui.ipynb`.

1. Open the chatbot UI tab.
2. **Log out** of the current session, then **log back in** as `gold_customer` (or any test user). Logging out forces a fresh token and a clean conversation, so prior un-masked responses cached in memory don't appear.
3. Send a prompt that pulls customer PII through a tool call, for example:

   > *Show me all my orders, including the email address and shipping address on each one.*

4. In the chatbot's reply, confirm that:
   - Email addresses are **not shown** in the response.
   - Shipping addresses are **not shown** in the response.
   - Order IDs, product names, statuses, and dates remain intact.

If real PII still shows up, check CloudWatch logs for `/aws/lambda/anycompany-pii-interceptor` to confirm the interceptor was invoked and the Bedrock Guardrail call succeeded.

## Summary

| Resource | Name | Purpose |
|----------|------|---------|
| IAM Role | `AnyCompanyPIIInterceptorLambdaRole` | Execution role for the interceptor Lambda |
| Lambda | `anycompany-pii-interceptor` | Applies Bedrock Guardrail to every tool response |
| Gateway update | `anycompany-agent-gateway` | Interceptor attached as a RESPONSE interception point |

PII fields (email, address, phone, credit card, SSN) are now anonymized in all tool responses before they reach the Harness. The Guardrail ID is reused from L2 — no new guardrail is created.

---
## Cleanup

Run the following cell to remove the interceptor and delete all resources created in this notebook.

**Resources deleted:**
- AgentCore Gateway: interceptor configuration (gateway resource itself stays — owned by L3 #1)
- Compute: interceptor Lambda function and its CloudWatch log group
- IAM: interceptor execution role + revert of gateway role mutation

> Run this cleanup **before** the L3 #1 (`1_setup_resources.ipynb`) cleanup so the Gateway IAM role mutation is reverted before the role is deleted.
> The Bedrock Guardrail is shared with L2 and is cleaned up by `1_pluggable_inference_layer.ipynb` cleanup — **not here**.

Uncomment and run to remove the interceptor and delete all resources created in this notebook.

In [ ]:
## === CLEANUP: Delete all resources created in this notebook ===
## ⚠️ Uncomment and run this cell only after you are done with ALL notebooks.
## NOTE: Run this BEFORE the L3 #1 (1_setup_resources.ipynb) cleanup so the
##       Gateway IAM role mutation is reverted before the role is deleted.
##       The Bedrock Guardrail is shared with L2 and is cleaned up by
##       `1_pluggable_inference_layer.ipynb` cleanup — NOT here.

# import boto3, json, os
#
# REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
# SSM_PREFIX = "/anycompany/agentcore"
#
# INTERCEPTOR_LAMBDA_NAME = "anycompany-pii-interceptor"
# INTERCEPTOR_ROLE_NAME   = "AnyCompanyPIIInterceptorLambdaRole"
# INTERCEPTOR_LOG_GROUP   = f"/aws/lambda/{INTERCEPTOR_LAMBDA_NAME}"
# GATEWAY_POLICY_NAME     = "AnyCompanyGatewayPermissions"  # owned by L3 #1
#
# iam               = boto3.client("iam")
# ssm               = boto3.client("ssm", region_name=REGION)
# lambda_client     = boto3.client("lambda", region_name=REGION)
# logs_client       = boto3.client("logs", region_name=REGION)
# agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)
# sts               = boto3.client("sts", region_name=REGION)
#
# ACCOUNT_ID = sts.get_caller_identity()["Account"]
# INTERCEPTOR_LAMBDA_ARN = f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:{INTERCEPTOR_LAMBDA_NAME}"
#
# def _ssm_get(key):
#     try:
#         return ssm.get_parameter(Name=f"{SSM_PREFIX}/{key}")["Parameter"]["Value"]
#     except ssm.exceptions.ParameterNotFound:
#         return None
#
# # ── Phase 1: Remove interceptor from Gateway ────────────────────────────────
# # update_gateway requires name, roleArn, authorizerType (and authorizerConfiguration
# # for CUSTOM_JWT). All are owned by L3 #1 and still in SSM if L3 #1 cleanup
# # has not yet run. If L3 #1 ran first, Gateway is gone — skip gracefully.
# gateway_id   = _ssm_get("gateway_id")
# gateway_name = _ssm_get("gateway_name")
# gateway_role = _ssm_get("gateway_role_arn")
# cog_pool     = _ssm_get("cognito_user_pool_id")
# cog_client   = _ssm_get("cognito_client_id")
#
# if all([gateway_id, gateway_name, gateway_role, cog_pool, cog_client]):
#     try:
#         agentcore_control.update_gateway(
#             gatewayIdentifier=gateway_id,
#             name=gateway_name,
#             roleArn=gateway_role,
#             authorizerType="CUSTOM_JWT",
#             authorizerConfiguration={
#                 "customJWTAuthorizer": {
#                     "discoveryUrl": f"https://cognito-idp.{REGION}.amazonaws.com/{cog_pool}/.well-known/openid-configuration",
#                     "allowedAudience": [cog_client],
#                 }
#             },
#             interceptorConfigurations=[],
#         )
#         print("Removed interceptor from Gateway.")
#     except agentcore_control.exceptions.ResourceNotFoundException:
#         print("Gateway already deleted; skipping interceptor detach.")
#     except Exception as e:
#         print(f"Update gateway: {e}")
# else:
#     print("Gateway/Cognito SSM keys missing; skipping interceptor detach (L3 #1 likely cleaned up first).")
#
# # ── Phase 2: Revert Gateway IAM role mutation (remove interceptor ARN) ──────
# if gateway_role:
#     gw_role_name = gateway_role.split("/")[-1]
#     try:
#         existing = iam.get_role_policy(RoleName=gw_role_name, PolicyName=GATEWAY_POLICY_NAME)["PolicyDocument"]
#         changed = False
#         for stmt in existing.get("Statement", []):
#             if stmt.get("Sid") != "InvokeLambdaTargets":
#                 continue
#             resources = stmt["Resource"] if isinstance(stmt["Resource"], list) else [stmt["Resource"]]
#             if INTERCEPTOR_LAMBDA_ARN in resources:
#                 resources = [r for r in resources if r != INTERCEPTOR_LAMBDA_ARN]
#                 stmt["Resource"] = resources
#                 changed = True
#             break
#         if changed:
#             iam.put_role_policy(
#                 RoleName=gw_role_name,
#                 PolicyName=GATEWAY_POLICY_NAME,
#                 PolicyDocument=json.dumps(existing),
#             )
#             print(f"Reverted gateway role mutation: removed {INTERCEPTOR_LAMBDA_ARN} from InvokeLambdaTargets.")
#         else:
#             print("Gateway role mutation already reverted (interceptor ARN not present).")
#     except iam.exceptions.NoSuchEntityException:
#         print("Gateway role/policy already deleted; skipping mutation revert.")
#     except Exception as e:
#         print(f"Gateway role revert: {e}")
# else:
#     print("gateway_role_arn SSM key missing; skipping mutation revert.")
#
# # ── Phase 3: Delete interceptor Lambda ──────────────────────────────────────
# try:
#     lambda_client.delete_function(FunctionName=INTERCEPTOR_LAMBDA_NAME)
#     print(f"Deleted Lambda: {INTERCEPTOR_LAMBDA_NAME}")
# except lambda_client.exceptions.ResourceNotFoundException:
#     print(f"Lambda {INTERCEPTOR_LAMBDA_NAME} already deleted.")
# except Exception as e:
#     print(f"Lambda delete: {e}")
#
# # ── Phase 4: Delete Lambda CloudWatch log group ─────────────────────────────
# try:
#     logs_client.delete_log_group(logGroupName=INTERCEPTOR_LOG_GROUP)
#     print(f"Deleted log group: {INTERCEPTOR_LOG_GROUP}")
# except logs_client.exceptions.ResourceNotFoundException:
#     pass
# except Exception as e:
#     print(f"Log group delete ({INTERCEPTOR_LOG_GROUP}): {e}")
#
# # ── Phase 5: Delete interceptor IAM role ────────────────────────────────────
# try:
#     for ap in iam.list_attached_role_policies(RoleName=INTERCEPTOR_ROLE_NAME).get("AttachedPolicies", []):
#         iam.detach_role_policy(RoleName=INTERCEPTOR_ROLE_NAME, PolicyArn=ap["PolicyArn"])
#     for p in iam.list_role_policies(RoleName=INTERCEPTOR_ROLE_NAME).get("PolicyNames", []):
#         iam.delete_role_policy(RoleName=INTERCEPTOR_ROLE_NAME, PolicyName=p)
#     iam.delete_role(RoleName=INTERCEPTOR_ROLE_NAME)
#     print(f"Deleted IAM role: {INTERCEPTOR_ROLE_NAME}")
# except iam.exceptions.NoSuchEntityException:
#     print(f"IAM role {INTERCEPTOR_ROLE_NAME} already deleted.")
# except Exception as e:
#     print(f"IAM role cleanup: {e}")
#
# print("\n✅ Sensitive data masking resources cleaned up.")